# Terminal Case Study — Phase 4: Scenario Comparison

**Case study**: Intermodal Container Terminal | **Phase**: 4 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Design a factorial experiment varying gate and crane counts.
2. Identify system bottlenecks using per-stage utilisation.
3. Apply paired-t confidence intervals to rank scenarios.
4. Formulate a cost-performance trade-off recommendation.

---
> Phase 4 answers: *"How many gates and cranes minimise truck turn-around time for a given budget?"*

In [ ]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from scipy import stats

from simdes.models.terminal import TerminalModel, TerminalParams
from simdes.analysis import confidence_interval

## Experimental Design

| Factor | Levels |
|---|---|
| `n_gates` | 1, 2, 3, 4 |
| `n_cranes` | 1, 2, 3 |

**4 × 3 = 12** scenarios.  Response: `mean_total_time`.  
CRN: all scenarios share the same set of 30 replication seeds.

In [ ]:
BASE = dict(arrival_rate=10.0, gate_mean=5.0, crane_mean=8.0, sim_time=8.0)
N_REPS    = 30
BASE_SEED = 2024

gates_levels  = [1, 2, 3, 4]
cranes_levels = [1, 2, 3]

rows = []
for n_gates, n_cranes in itertools.product(gates_levels, cranes_levels):
    p = TerminalParams(n_gates=n_gates, n_cranes=n_cranes, **BASE)
    model = TerminalModel(params=p, seed=BASE_SEED)
    df_sc = model.run_replications(N_REPS)
    m, lo, hi = confidence_interval(df_sc['mean_total_time'].to_numpy())
    rows.append(dict(n_gates=n_gates, n_cranes=n_cranes,
                     mean=m, ci_lo=lo, ci_hi=hi,
                     raw=df_sc['mean_total_time'].to_numpy(),
                     mean_wait_gate=df_sc['mean_wait_gate'].mean(),
                     mean_wait_crane=df_sc['mean_wait_crane'].mean()))

results = pd.DataFrame(rows)
results[['n_gates', 'n_cranes', 'mean', 'ci_lo', 'ci_hi',
         'mean_wait_gate', 'mean_wait_crane']]

In [ ]:
# Heat map — mean total time
pivot = results.pivot(index='n_cranes', columns='n_gates', values='mean')

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r')
ax.set_xticks(range(len(gates_levels)))
ax.set_xticklabels([f'{g} gate(s)' for g in gates_levels])
ax.set_yticks(range(len(cranes_levels)))
ax.set_yticklabels([f'{c} crane(s)' for c in cranes_levels])
for i, j in itertools.product(range(len(cranes_levels)), range(len(gates_levels))):
    ax.text(j, i, f'{pivot.values[i,j]:.1f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, label='Mean total time (min)')
ax.set_xlabel('Gates')
ax.set_ylabel('Cranes')
ax.set_title('Terminal scenarios — mean truck turn-around time')
fig.tight_layout()
plt.show()

In [ ]:
# Bottleneck analysis: gate wait vs crane wait for each scenario
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for i, nc in enumerate(cranes_levels):
    sub = results[results.n_cranes == nc].sort_values('n_gates')
    ax = axes[i]
    ax.bar(sub['n_gates'] - 0.2, sub['mean_wait_gate'],  0.35, label='Gate wait',  color='tab:blue',   alpha=0.7)
    ax.bar(sub['n_gates'] + 0.2, sub['mean_wait_crane'], 0.35, label='Crane wait', color='tab:orange',  alpha=0.7)
    ax.set_xlabel('Number of gates')
    ax.set_title(f'{nc} crane(s)')
    if i == 0:
        ax.set_ylabel('Mean wait (min)')
        ax.legend()
    ax.set_xticks(gates_levels)
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('Per-stage wait by scenario')
fig.tight_layout()
plt.show()

In [ ]:
# Paired-t: baseline (2 gates, 1 crane) vs. best scenario
baseline_row = results[(results.n_gates == 2) & (results.n_cranes == 1)].iloc[0]
best_row     = results.loc[results['mean'].idxmin()]

diff = baseline_row['raw'] - best_row['raw']
_, p_value = stats.ttest_rel(baseline_row['raw'], best_row['raw'])
d_mean, d_lo, d_hi = confidence_interval(diff)

print(f"Baseline  (gates=2, cranes=1): {baseline_row['mean']:.2f} min")
print(f"Best      (gates={best_row.n_gates}, cranes={best_row.n_cranes}):  {best_row['mean']:.2f} min")
print(f"Reduction: {d_mean:.2f} min  95% CI [{d_lo:.2f}, {d_hi:.2f}]")
print(f"p-value: {p_value:.4f}")

## Cost-Performance Analysis

Assume:
- Gate agent cost: \$40/hr
- Crane cost: \$120/hr
- Truck penalty: \$2/minute of turn-around time per truck

Compute total operating cost per day for each scenario.

In [ ]:
GATE_COST  = 40.0   # $/hr
CRANE_COST = 120.0  # $/hr
TRUCK_PEN  = 2.0    # $/min per truck
SIM_HRS    = 8.0

results['staff_cost'] = (results['n_gates'] * GATE_COST + results['n_cranes'] * CRANE_COST) * SIM_HRS
# Approximate n_trucks ~= arrival_rate * SIM_HRS
n_trucks_est = 10.0 * SIM_HRS
results['truck_cost'] = results['mean'] * TRUCK_PEN * n_trucks_est
results['total_cost'] = results['staff_cost'] + results['truck_cost']

best_cost_row = results.loc[results['total_cost'].idxmin()]
print('Lowest total cost scenario:')
print(best_cost_row[['n_gates', 'n_cranes', 'mean', 'staff_cost', 'truck_cost', 'total_cost']])

## Try It Yourself

1. Sensitivity analysis: change the truck penalty to \$5/min. Does the optimal scenario change?
2. What arrival rate would make 4 gates necessary to stay below Wq_gate < 3 min?
3. Add a third factor: `gate_mean` ∈ {4, 5, 6} min. Does this interact with crane count?